# 4. Feature engineering

Models can't read text, so the messages have to be turned into numbers. Here I split the data, try CountVectorizer and TF-IDF, and decide which one to use for the models.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('data/spam_clean.csv')
X = df['clean_message']
y = df['label_num']

## Train/test split

The split comes **before** any vectorizer is fitted, so nothing from the test set leaks into the features. `stratify=y` keeps the same spam percentage in both parts, and `random_state` makes the split the same every time.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape[0], 'messages, spam share:', round(y_train.mean(), 3))
print('Test :', X_test.shape[0], 'messages, spam share:', round(y_test.mean(), 3))

Train: 4100 messages, spam share: 0.123
Test : 1026 messages, spam share: 0.123


## CountVectorizer

It builds a vocabulary of all the words in the training messages, and then represents each message as a row of word counts. `fit_transform` learns the vocabulary from the training data, and the test data only gets `transform`.

In [4]:
count_vec = CountVectorizer()
X_train_count = count_vec.fit_transform(X_train)
X_test_count = count_vec.transform(X_test)

print('Train matrix:', X_train_count.shape)
print('Test matrix :', X_test_count.shape)
print('Vocabulary size:', len(count_vec.vocabulary_))

cells = X_train_count.shape[0] * X_train_count.shape[1]
print('Non-zero values:', round(X_train_count.nnz / cells * 100, 2), '%')

Train matrix: (4100, 7487)
Test matrix : (1026, 7487)
Vocabulary size: 7487
Non-zero values: 0.17 %


The test matrix has the same columns as the train matrix. Words that only appear in the test messages are simply ignored.

Most of the matrix is zeros, because a message only uses a few words from the whole vocabulary. Here is one message and its non-zero counts:

In [5]:
words = count_vec.get_feature_names_out()
row = X_train_count[0].toarray()[0]

print(X_train.iloc[0])
print()
for i in row.nonzero()[0]:
    print(words[i], row[i])

are you willing to go for apps class

apps 1
are 1
class 1
for 1
go 1
to 1
willing 1
you 1


## TF-IDF

Same idea, but instead of raw counts each word gets a weight. Words that appear in many messages (like "the" or "you") get a lower weight, and rarer words get a higher one.

In [6]:
tfidf_vec = TfidfVectorizer()
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

print('Train matrix:', X_train_tfidf.shape)
print('Test matrix :', X_test_tfidf.shape)

Train matrix: (4100, 7487)
Test matrix : (1026, 7487)


In [7]:
# same message as above, with the tf-idf weights
tfidf_words = tfidf_vec.get_feature_names_out()
row = X_train_tfidf[0].toarray()[0]

for i in row.nonzero()[0]:
    print(tfidf_words[i], round(row[i], 3))

apps 0.559
are 0.246
class 0.407
for 0.226
go 0.283
to 0.155
willing 0.531
you 0.156


## Which one should I use?

To pick one without touching the test set, I run a 5-fold cross-validation on the **training data only**. Each candidate is a Pipeline (vectorizer + model), so inside every fold the vectorizer is fitted only on that fold's training part.

I'm also checking two other things here:
- Stopword removal (`stop_words='english'`), which I skipped in preprocessing
- `class_weight='balanced'` for Logistic Regression, because only about 12% of the messages are spam

The score is F1 for the spam class.

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

candidates = {
    'Count + Naive Bayes': Pipeline([
        ('vec', CountVectorizer()), ('model', MultinomialNB())]),
    'Count + LogReg': Pipeline([
        ('vec', CountVectorizer()), ('model', LogisticRegression(max_iter=1000))]),
    'Count + LogReg (balanced)': Pipeline([
        ('vec', CountVectorizer()), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
    'TF-IDF + Naive Bayes': Pipeline([
        ('vec', TfidfVectorizer()), ('model', MultinomialNB())]),
    'TF-IDF + LogReg': Pipeline([
        ('vec', TfidfVectorizer()), ('model', LogisticRegression(max_iter=1000))]),
    'TF-IDF + LogReg (balanced)': Pipeline([
        ('vec', TfidfVectorizer()), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
    'Count no stopwords + Naive Bayes': Pipeline([
        ('vec', CountVectorizer(stop_words='english')), ('model', MultinomialNB())]),
    'Count no stopwords + LogReg (balanced)': Pipeline([
        ('vec', CountVectorizer(stop_words='english')),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
}

cv_scores = {}
for name, pipe in candidates.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    cv_scores[name] = round(scores.mean(), 4)

pd.DataFrame({'CV F1': cv_scores})

,CV F1
Count + Naive Bayes,0.9351
Count + LogReg,0.9129
Count + LogReg (balanced),0.9321
TF-IDF + Naive Bayes,0.7469
TF-IDF + LogReg,0.8267
TF-IDF + LogReg (balanced),0.9160
Count no stopwords + Naive Bayes,0.9352
Count no stopwords + LogReg (balanced),0.9269


## Result

- CountVectorizer beats TF-IDF for both models. TF-IDF + Naive Bayes is by far the weakest (0.75). I didn't tune anything, so this only tells me about the default settings. My guess is that the default smoothing in Naive Bayes is too strong for the small tf-idf values.
- `class_weight='balanced'` helps Logistic Regression a lot (0.91 to 0.93 with counts, 0.83 to 0.92 with tf-idf), so I'll use it.
- Removing stopwords doesn't help. Naive Bayes stays the same (0.935) and Logistic Regression gets slightly worse, so I'm leaving the stopwords in.

Decision for the next notebook: **CountVectorizer without stopword removal**, with Naive Bayes and a balanced Logistic Regression.

## Saving the split

The next notebook builds its own Pipelines, so the vectorizer gets fitted on the training text again there. That means I don't need to save the matrices, only the train and test text.

In [9]:
train = pd.DataFrame({'clean_message': X_train, 'label_num': y_train})
test = pd.DataFrame({'clean_message': X_test, 'label_num': y_test})

train.to_csv('data/train.csv', index=False)
test.to_csv('data/test.csv', index=False)
print(train.shape, test.shape)

(4100, 2) (1026, 2)
